# Week 2 · Backpropagation from scratch

**Question.** How does a binary classification error reach the weights in a hidden layer?

Use the project's Python 3.11 Jupyter kernel and installed NumPy and Matplotlib. Start after Week 1's affine score and binary cross-entropy. You will trace a two-layer network, verify its gradients, and compare it with a linear classifier on synthetic research-triage data. If CPU PyTorch is available in this kernel, the final comparison also runs autograd on the identical model and training split.

The four parameter shapes use $d$ input features and $h$ hidden units: $W_1$ is $(d,h)$, $b_1$ is $(h,)$, $W_2$ is $(h,1)$, and $b_2$ is $(1,)$. For a batch of $n$ rows, compute

$$A=XW_1+b_1,\quad H=\max(0,A),\quad Z=HW_2+b_2,\quad
L=\operatorname{mean}\operatorname{softplus}((1-2Y)Z).$$

Both $Y$ and $Z$ have shape $(n,1)$. The mean divides by $n$ once. A batch forward/backward pass has leading time cost $O(ndh)$ and uses $O(nh)$ activation storage. The code assumes finite float arrays, a nonempty batch, and labels in $\{0,1\}$.

In [ ]:
from importlib.util import find_spec
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray

Array = NDArray[np.float64]


class Parameters(NamedTuple):
    w1: Array
    b1: Array
    w2: Array
    b2: Array


def forward(x: Array, params: Parameters) -> tuple[Array, Array, Array]:
    """Return hidden preactivations, hidden activations, and column logits."""
    a = x @ params.w1 + params.b1
    h = np.maximum(a, 0.0)
    return a, h, h @ params.w2 + params.b2


def binary_loss(logits: Array, y: Array) -> float:
    """Return mean binary cross-entropy from matching column logits and labels."""
    if logits.shape != y.shape or logits.ndim != 2 or logits.shape[1] != 1:
        raise ValueError("Logits and labels must have identical (n, 1) shapes")
    return float(np.logaddexp(0.0, (1.0 - 2.0 * y) * logits).mean())


def sigmoid(z: Array) -> Array:
    """Compute a stable sigmoid for an array of logits."""
    return np.exp(-np.logaddexp(0.0, -z))


def backward(x: Array, y: Array, params: Parameters) -> tuple[float, Parameters]:
    """Return the loss and its mean gradients in parameter order."""
    a, h, z = forward(x, params)
    loss = binary_loss(z, y)
    dz = (sigmoid(z) - y) / len(y)
    da = (dz @ params.w2.T) * (a > 0)
    return loss, Parameters(x.T @ da, da.sum(axis=0), h.T @ dz, dz.sum(axis=0))

## Worked trace and gradient check

Take $x=(1,2)$ and $y=1$. The example below gives $A=(0.4,-0.1)$,
$H=(0.4,0)$, and $Z\approx0$. The loss is $\log 2$ and $dZ=-0.5$.
The incoming gradient on the first hidden coordinate is $-0.5(1.5)$.
ReLU blocks the second coordinate. Verify all four gradients and then
take one small step.

The centered difference $\bigl(L(\theta+\varepsilon)-L(\theta-\varepsilon)\bigr)/(2\varepsilon)$ checks a coordinate independently of the backward equations. Check the smooth region first. A check near a ReLU kink can disagree with the chosen subgradient even if the implementation is sound.

In [ ]:
trace_x = np.array([[1.0, 2.0]])
trace_y = np.array([[1.0]])
trace = Parameters(
    np.array([[0.2, -0.4], [0.1, 0.1]]),
    np.array([0.0, 0.1]),
    np.array([[1.5], [-0.5]]),
    np.array([-0.6]),
)
trace_loss, trace_grad = backward(trace_x, trace_y, trace)
np.testing.assert_allclose(trace_loss, np.log(2.0))
np.testing.assert_allclose(trace_grad.w1, [[-0.75, 0.0], [-1.5, 0.0]])
np.testing.assert_allclose(trace_grad.b1, [-0.75, 0.0])
np.testing.assert_allclose(trace_grad.w2, [[-0.2], [0.0]])
np.testing.assert_allclose(trace_grad.b2, [-0.5])
updated_trace = Parameters(*(p - 0.1 * g for p, g in zip(trace, trace_grad)))
assert binary_loss(forward(trace_x, updated_trace)[-1], trace_y) < trace_loss


def central_differences(
    x: Array, y: Array, params: Parameters, step: float = 1e-5
) -> Parameters:
    """Estimate each parameter coordinate on a small fixed batch."""
    estimates = []
    for block_index, block in enumerate(params):
        estimated = np.zeros_like(block)
        for index in np.ndindex(block.shape):
            plus = list(params)
            minus = list(params)
            plus[block_index] = block.copy()
            minus[block_index] = block.copy()
            plus[block_index][index] += step
            minus[block_index][index] -= step
            loss_plus = binary_loss(forward(x, Parameters(*plus))[-1], y)
            loss_minus = binary_loss(forward(x, Parameters(*minus))[-1], y)
            estimated[index] = (loss_plus - loss_minus) / (2.0 * step)
        estimates.append(estimated)
    return Parameters(*estimates)


check_x = np.array([[1.0, 2.0], [-0.5, 0.7], [0.8, -1.1]])
check_y = np.array([[1.0], [0.0], [1.0]])
assert np.min(np.abs(forward(check_x, trace)[0])) > 1e-3
_, analytic = backward(check_x, check_y, trace)
numeric = central_differences(check_x, check_y, trace)
for actual, estimate in zip(analytic, numeric):
    np.testing.assert_allclose(actual, estimate, rtol=1e-5, atol=1e-7)
gradient_error = max(float(np.max(np.abs(a - b))) for a, b in zip(analytic, numeric))
print(f"Maximum central-difference error: {gradient_error:.3e}")

## Controlled experiment · A nonlinear review rule

The synthetic inputs represent a standardized surprise and a contemporaneously available market move. The target is disagreement in sign, with 5% independent label flips. The label is a toy review rule. It describes neither future returns nor analyst behavior.

Generate 512 rows in timestamp-index order using seed 2026. Fit on the first 256 rows, inspect validation on the next 128, and keep the final 128 out of this experiment. The linear baseline and two-layer network use the same rows, mean loss, learning rate, and number of full-batch steps. They have different parameterizations and initializations. Keep the validation comparison descriptive, and hold the split fixed for any follow-up.

In [ ]:
rng = np.random.default_rng(2026)
features = rng.uniform(-1.0, 1.0, size=(512, 2))
review = features[:, :1] * features[:, 1:] < 0.0
flips = rng.random((512, 1)) < 0.05
labels = np.logical_xor(review, flips).astype(np.float64)

x_train, y_train = features[:256], labels[:256]
x_val, y_val = features[256:384], labels[256:384]
epochs, rate = 450, 0.3
initial = Parameters(
    rng.normal(0.0, 0.5, (2, 8)),
    np.zeros(8),
    rng.normal(0.0, 0.35, (8, 1)),
    np.zeros(1),
)


def fit_linear() -> tuple[Array, Array, Array]:
    """Train the Week 1 logistic baseline on this experiment's training split."""
    weights = np.zeros((2, 1))
    bias = np.zeros(1)
    history = []
    for step in range(epochs + 1):
        train_z = x_train @ weights + bias
        val_z = x_val @ weights + bias
        history.append((binary_loss(train_z, y_train), binary_loss(val_z, y_val)))
        if step < epochs:
            dz = (sigmoid(train_z) - y_train) / len(y_train)
            weights -= rate * (x_train.T @ dz)
            bias -= rate * dz.sum(axis=0)
    return weights, bias, np.array(history)


def fit_numpy() -> tuple[Parameters, Array]:
    """Train the hidden-layer model under the same full-batch protocol."""
    params = initial
    history = []
    for step in range(epochs + 1):
        train_loss, grads = backward(x_train, y_train, params)
        history.append((train_loss, binary_loss(forward(x_val, params)[-1], y_val)))
        if step < epochs:
            params = Parameters(*(p - rate * g for p, g in zip(params, grads)))
    return params, np.array(history)


linear_w, linear_b, linear_history = fit_linear()
numpy_params, numpy_history = fit_numpy()
assert numpy_history[-1, 0] < numpy_history[0, 0]
assert np.isfinite(numpy_history).all()

for name, history, val_logits in [
    ("Linear", linear_history, x_val @ linear_w + linear_b),
    ("Two-layer NumPy", numpy_history, forward(x_val, numpy_params)[-1]),
]:
    accuracy = np.mean((val_logits > 0.0) == y_val)
    print(
        f"{name}: training BCE {history[0, 0]:.4f} -> {history[-1, 0]:.4f}, "
        f"validation BCE {history[-1, 1]:.4f}, validation accuracy {accuracy:.3f}"
    )

figure, axes = plt.subplots(1, 2, figsize=(9, 3.3), layout="constrained")
for index, (axis, title) in enumerate(zip(axes, ["Training", "Validation"])):
    axis.plot(linear_history[:, index], label="Linear")
    axis.plot(numpy_history[:, index], label="Two-layer NumPy")
    axis.set(title=title, xlabel="Full-batch updates", ylabel="Mean BCE (nats)")
    axis.legend()
plt.show()

## Library comparison · CPU PyTorch autograd

The code uses identical initial weights, float64 arrays, inputs, and mean loss. It compares the initial gradients, all recorded training and validation losses, and final predictions. Both implementations use plain gradient descent. The NumPy lab can still run if PyTorch is absent. See the [official PyTorch setup selector](https://pytorch.org/get-started/locally/) before adding it to the environment used by this kernel.

In [ ]:
if find_spec("torch") is None:
    print("PyTorch is unavailable in this kernel. NumPy and finite-difference checks ran.")
else:
    import torch
    from torch.nn.functional import binary_cross_entropy_with_logits

    tx, ty, vx, vy = (
        torch.tensor(values, dtype=torch.float64, device="cpu")
        for values in (x_train, y_train, x_val, y_val)
    )
    params = [
        torch.tensor(values, dtype=torch.float64, device="cpu", requires_grad=True)
        for values in initial
    ]

    def torch_logits(x, parameters):
        """Apply the same affine, ReLU, and affine operations on CPU."""
        w1, b1, w2, b2 = parameters
        return torch.relu(x @ w1 + b1) @ w2 + b2

    torch_history = []
    for step in range(epochs + 1):
        train_loss = binary_cross_entropy_with_logits(torch_logits(tx, params), ty)
        with torch.no_grad():
            val_loss = binary_cross_entropy_with_logits(torch_logits(vx, params), vy)
        torch_history.append((train_loss.item(), val_loss.item()))
        if step == epochs:
            break
        train_loss.backward()
        if step == 0:
            _, expected = backward(x_train, y_train, initial)
            for param, grad in zip(params, expected):
                np.testing.assert_allclose(param.grad.detach().numpy(), grad, atol=1e-10)
        with torch.no_grad():
            for param in params:
                param -= rate * param.grad
                param.grad = None

    np.testing.assert_allclose(torch_history, numpy_history, rtol=1e-6, atol=1e-8)
    np.testing.assert_allclose(
        torch_logits(vx, params).detach().numpy(),
        forward(x_val, numpy_params)[-1],
        rtol=1e-6,
        atol=1e-8,
    )
    print(f"CPU PyTorch {torch.__version__}: initial gradients, curves and logits agree.")

## Independent work

1. Build your own three-row batch with both labels and a hidden unit that is inactive on only some rows. Pick one entry of each weight matrix and predict the sign of its gradient before running code. Use a central difference to check every parameter block and document the tolerance.
2. Diagnose what happens when the mean is divided by the batch size a second time. Test the buggy version on a batch with more than one row and at least one nonzero gradient. Describe which check identifies it.
3. Vary one factor, such as hidden width or noise. State a prediction, keep the split and training budget fixed, and compare validation loss and one error case. Report the single-run limitation. If you plan to use the untouched test block later, fix your selection rule before evaluating it.

One synthetic path illustrates the representation challenge. It does not provide a sample of investment outcomes. A real investigation would check when both features were available, use a chronological and issuer-aware holdout, and measure review burden alongside missed cases.

**Reading:** [Prince, Understanding Deep Learning](https://udlbook.github.io/udlbook/), loss and gradient excerpts, and [AIMA, third edition](https://aima.cs.berkeley.edu/3rd-ed/contents.html), §§18.4 and 18.7.